<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/09-Neural-Networks/Guided-Project/GP09_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GP09: Neural Networks in Action
### CAI1001C: Artificial Intelligence (AI) Thinking | Chapter 9
**Guided Project — Reference Material (Not Graded)**

This notebook is your in-class demo and reference. It covers two pipelines:
- **Part 1 — MNIST Handwritten Digits**: Build and train your first neural network on image data
- **Part 2 — Bank Customer Churn**: Compare a neural network against all four classical classifiers from Chapters 7–8

Keep this notebook — you'll reference it for your homework assignment.


## Learning Objectives

By the end of this guided project, you will be able to:
1. Load and explore image data (MNIST) and understand pixels as numbers
2. Build and train a neural network using TensorFlow/Keras
3. Modify the network's architecture (layers, neurons) and observe the impact on accuracy
4. Modify the number of training epochs and identify signs of overfitting
5. Compare neural network performance against classical classifiers on the same dataset


In [ ]:
# ============================================
# Run this cell first — imports and setup
# ============================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print("All imports successful — ready to go!")

---
# Part 1: MNIST Handwritten Digit Recognition

MNIST is the most famous dataset in machine learning — 70,000 images of handwritten digits (0–9), each a 28×28 pixel grayscale image. Every pixel is a number from 0 (black) to 255 (white).

We're going to load this data, understand what images look like to a computer, and then build a neural network that learns to read handwriting.


## Example 9.1: Loading and Exploring MNIST

Our first step is understanding images as data. Each image is just a grid of numbers — and those numbers are the features our neural network will learn from.


In [ ]:
# ============================================
# Example 9.1: Loading and Exploring MNIST
# Purpose: Understand images as numerical data
# ============================================

# Step 1: Load the dataset (built into Keras — no download needed)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Step 2: Explore the data shape
print(f"Training images: {X_train.shape}")     # 60,000 images, each 28x28
print(f"Test images:     {X_test.shape}")       # 10,000 images for evaluation
print(f"Each image is {X_train.shape[1]}x{X_train.shape[2]} pixels")
print(f"Pixel value range: {X_train.min()} to {X_train.max()}")
print(f"First 10 labels: {y_train[:10]}")

**What just happened:** We loaded 60,000 training images and 10,000 test images. Each image is a 28×28 grid of pixel values from 0 to 255. The labels tell us which digit each image represents (0–9).


In [ ]:
# Let's see what some of these digits look like
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}", fontsize=12)
    ax.axis('off')
plt.suptitle("First 8 Training Images", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# What does the computer actually see? Numbers.
print("First image (the digit 5) as raw pixel values:")
print(X_train[0])
print(f"\nShape: {X_train[0].shape} — a 28x28 grid of numbers")

### ▶ Your Turn

Look at the pixel grid printed above. Values near 0 are black (background), values near 255 are white (the digit stroke).

**Try this:** Change `X_train[0]` to `X_train[1]` in the cell above and re-run. What digit is it? Can you see the shape in the numbers?


In [ ]:
# Your exploration space


---
## Example 9.2: Build, Train, and Evaluate a Neural Network

Now we build our first neural network. The architecture:
- **Input**: 784 numbers (28×28 pixels, flattened)
- **Hidden layer**: 128 neurons with ReLU activation
- **Output**: 10 neurons with softmax (one per digit 0–9)

We'll normalize pixel values to 0–1 first — neural networks train better on small, consistent number ranges.


In [ ]:
# ============================================
# Example 9.2: Build, Train, and Evaluate
# Purpose: First complete neural network pipeline
# ============================================

# Step 1: Normalize pixel values from 0-255 to 0-1
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0
print(f"Pixel range after normalization: {X_train_norm.min()} to {X_train_norm.max()}")

# Step 2: Build the model
tf.random.set_seed(42)
np.random.seed(42)

model_1 = Sequential([
    Flatten(input_shape=(28, 28)),        # Flatten 28x28 → 784
    Dense(128, activation='relu'),         # Hidden layer: 128 neurons
    Dense(10, activation='softmax')        # Output layer: 10 digits
])

# Step 3: Compile — how should the model learn?
model_1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Step 4: See the architecture
model_1.summary()

**101,770 trainable parameters!** That's 100,480 weights connecting 784 inputs to 128 neurons, plus 1,290 connecting the hidden layer to the 10 outputs. Every one starts as a random number and gets adjusted during training.

Now let's train:


In [ ]:
# Step 5: Train for 5 epochs
print("Training Model 1 (1 hidden layer, 128 neurons, 5 epochs)...")
history_1 = model_1.fit(
    X_train_norm, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

# Step 6: Evaluate on test data (never seen during training)
test_loss, test_acc = model_1.evaluate(X_test_norm, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc*100:.2f}%")

**97.08% accuracy!** Out of 10,000 test images, the model correctly identified over 9,700 handwritten digits.

Let's see its predictions:


In [ ]:
# Step 7: See predictions vs. actual
predictions = model_1.predict(X_test_norm[:20], verbose=0)
pred_labels = np.argmax(predictions, axis=1)

print(f"Predicted: {pred_labels}")
print(f"Actual:    {y_test[:20]}")

# Find mistakes
for i in range(20):
    if pred_labels[i] != y_test[i]:
        confidence = predictions[i][pred_labels[i]] * 100
        print(f"\nMistake at index {i}: predicted {pred_labels[i]} "
              f"(confidence: {confidence:.1f}%), actual was {y_test[i]}")

**19 out of 20 correct.** The model confused a 5 for a 6 at 84.6% confidence. Let's see that image:


In [ ]:
# Show the misclassified digit
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(X_test[8], cmap='gray')
ax.set_title(f"Predicted: 6 (84.6%)\nActual: 5", fontsize=12)
ax.axis('off')
plt.show()
print("Can you see why the model was confused? Some 5s and 6s look very similar in handwriting.")

### ▶ Your Turn — Modification Moment 1: More Epochs

**What to do:** Copy the training code below, but change `epochs=5` to `epochs=10`. Run it and answer:
1. Did test accuracy improve? (It should reach about 97.52%)
2. Look at the training output — by epoch 8-10, the training accuracy is ~99.6% but validation accuracy is ~97.3%. That gap is **overfitting** — the model is memorizing training data instead of learning general patterns.


In [ ]:
# Modification 1: Try 10 epochs
# Copy and modify the training code here

tf.random.set_seed(42)
np.random.seed(42)

model_3 = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model_3.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ▶ MODIFY: Change epochs from 5 to 10
history_3 = model_3.fit(
    X_train_norm, y_train,
    epochs=10,      # <-- Changed from 5 to 10
    batch_size=32,
    validation_split=0.1
)

test_loss_3, test_acc_3 = model_3.evaluate(X_test_norm, y_test, verbose=0)
print(f"\nTest accuracy (10 epochs): {test_acc_3*100:.2f}%")
print(f"\nNotice: training accuracy climbed toward 99.6%, but test accuracy")
print(f"only reached ~97.5%. The model started memorizing — that's overfitting.")

### ▶ Your Turn — Modification Moment 2: Add a Hidden Layer

**What to do:** Run the cell below. We've added a second hidden layer (64 neurons) to the original architecture. Does it improve accuracy?


In [ ]:
# Modification 2: Try 2 hidden layers
tf.random.set_seed(42)
np.random.seed(42)

model_2 = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),    # Hidden layer 1
    Dense(64, activation='relu'),     # Hidden layer 2 (NEW)
    Dense(10, activation='softmax')
])

model_2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_2 = model_2.fit(
    X_train_norm, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

test_loss_2, test_acc_2 = model_2.evaluate(X_test_norm, y_test, verbose=0)
print(f"\nTest accuracy (2 hidden layers, 5 epochs): {test_acc_2*100:.2f}%")

### MNIST Model Comparison

| Model | Architecture | Epochs | Test Accuracy |
|-------|-------------|--------|---------------|
| Model 1 | 128 neurons, 1 hidden layer | 5 | **97.08%** |
| Model 2 | 128 + 64 neurons, 2 hidden layers | 5 | **97.25%** |
| Model 3 | 128 neurons, 1 hidden layer | 10 | **97.52%** |

**Key takeaway:** More complexity helps — but with diminishing returns. On a clean problem like MNIST, a simple network already does great. The biggest gains come from the first layer and the first few epochs. After that, you risk overfitting more than improving.


---
# Part 2: Bank Customer Churn — Neural Network vs. Classical Classifiers

Now we test a neural network on the same kind of tabular data you've been working with since Chapter 6. The bank churn dataset has 10,000 customers with features like credit score, age, balance, and activity status. The target: did the customer leave the bank (churn)?

This is where it all comes together — five classifier types, same data, head-to-head comparison.


In [ ]:
# ============================================
# Load and Prepare the Bank Churn Data
# ============================================

# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/c-marq/AI-Thinking-CAI1001C/refs/heads/main/09-Neural-Networks/Datasets/bank_churn.csv')

print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Churn rate: {df['Exited'].mean():.2%}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Drop non-feature columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# Encode categorical variables
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# Split features and target
X = df.drop(columns=['Exited'])
y = df['Exited']

# Train/test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (essential for neural networks and SVMs)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

print(f"Training: {X_train_c.shape[0]} rows | Test: {X_test_c.shape[0]} rows")
print(f"Test set: {y_test_c.sum()} churned, {(y_test_c==0).sum()} stayed")
print(f"Test churn rate: {y_test_c.mean():.2%}")
print(f"Features: {list(X.columns)}")

## Train the Neural Network

We're building a 2-hidden-layer network: 64 neurons → 32 neurons → 1 output (sigmoid for binary classification: churned or stayed).


In [ ]:
# ============================================
# Train Neural Network on Churn Data
# ============================================
tf.random.set_seed(42)
np.random.seed(42)

nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')    # Binary: churned (1) or stayed (0)
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training neural network (2 hidden layers, 10 epochs)...")
nn_model.fit(
    X_train_scaled, y_train_c,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

# Evaluate
nn_preds = (nn_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
nn_acc = accuracy_score(y_test_c, nn_preds)
nn_prec = precision_score(y_test_c, nn_preds)
nn_rec = recall_score(y_test_c, nn_preds)

print(f"\nNeural Network Results:")
print(f"  Accuracy:  {nn_acc*100:.2f}%")
print(f"  Precision: {nn_prec*100:.2f}%")
print(f"  Recall:    {nn_rec*100:.2f}%")

## Train the Four Classical Classifiers

Same data, same split, same scaling. Let's see how the classifiers from Chapters 7–8 compare.


In [ ]:
# ============================================
# Train All Four Classical Classifiers
# ============================================

# Logistic Regression (Ch 8)
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train_c)

# k-NN (Ch 7)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train_c)

# Decision Tree (Ch 7)
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_scaled, y_train_c)

# SVM (Ch 8)
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_scaled, y_train_c)

print("All four classical classifiers trained!")

## The Grand Comparison

Now let's build the comparison table. The first two classifiers are done for you — **complete the remaining three.**


In [ ]:
# ============================================
# Grand Comparison: All 5 Classifier Types
# ============================================

print(f"{'Model':<30} {'Accuracy':<12} {'Precision':<12} {'Recall':<10}")
print("-" * 64)

# --- Logistic Regression (done for you) ---
lr_preds = lr.predict(X_test_scaled)
lr_acc = accuracy_score(y_test_c, lr_preds)
lr_prec = precision_score(y_test_c, lr_preds, zero_division=0)
lr_rec = recall_score(y_test_c, lr_preds)
print(f"{'Logistic Regression':<30} {lr_acc*100:.2f}%{'':<6} {lr_prec*100:.2f}%{'':<6} {lr_rec*100:.2f}%")

# --- k-NN (done for you) ---
knn_preds = knn.predict(X_test_scaled)
knn_acc = accuracy_score(y_test_c, knn_preds)
knn_prec = precision_score(y_test_c, knn_preds, zero_division=0)
knn_rec = recall_score(y_test_c, knn_preds)
print(f"{'k-NN (K=5)':<30} {knn_acc*100:.2f}%{'':<6} {knn_prec*100:.2f}%{'':<6} {knn_rec*100:.2f}%")

# --- Decision Tree --- YOUR CODE HERE ---
# Follow the same pattern: predict, calculate accuracy/precision/recall, print
dt_preds = dt.predict(X_test_scaled)
# YOUR CODE HERE: calculate dt_acc, dt_prec, dt_rec using accuracy_score, precision_score, recall_score


# --- SVM --- YOUR CODE HERE ---
# Follow the same pattern
svm_preds = svm.predict(X_test_scaled)
# YOUR CODE HERE: calculate svm_acc, svm_prec, svm_rec


print("-" * 64)

# --- Neural Network (done for you — already computed above) ---
print(f"{'Neural Network (2 layers)':<30} {nn_acc*100:.2f}%{'':<6} {nn_prec*100:.2f}%{'':<6} {nn_rec*100:.2f}%")

### Expected Grand Comparison Table

| Model | Accuracy | Precision | Recall | Churners Caught |
|-------|----------|-----------|--------|-----------------|
| Logistic Regression | 81.10% | 55.24% | 20.10% | 79 / 393 |
| k-NN (K=5) | 83.00% | 61.09% | 37.15% | 146 / 393 |
| Decision Tree (depth=4) | 85.35% | 76.60% | 36.64% | 144 / 393 |
| SVM (linear) | 80.35% | 0.00% | **0.00%** | **0 / 393** |
| **Neural Network (2 layers)** | **86.10%** | 74.47% | **44.53%** | **175 / 393** |

### Key Discussion Points

🔴 **The SVM caught ZERO churners.** It predicted every single customer would stay — and still got 80% accuracy. Why? Because 80% of customers *did* stay. A model that does literally nothing gets 80% accuracy. **This is why accuracy alone is a lie when classes are imbalanced.**

🟢 **The neural network won on recall** — catching 175 churners vs. the decision tree's 144. That's 31 more customers your retention team can reach. At $500/customer per year, that's $15,500 in saved revenue.

🟡 **But the accuracy gap is tiny** — the NN beat the decision tree by less than 1%. And the decision tree can *explain* every prediction. The neural network can't. Which would you deploy?


### ▶ Your Turn — Think About It

The decision tree at 85.35% can explain every decision: "This customer is over 45, has only one product, and is inactive — high churn risk."

The neural network at 86.10% caught 31 more churners but can't explain *any* of its decisions.

**Discussion question:** If you were the bank's data team, which model would you recommend to the retention department? What factors beyond accuracy would influence your decision? Discuss with the person next to you.


---
## What We Built Today

**Part 1 — MNIST:** You built a neural network that reads handwriting at 97% accuracy. You saw how adding layers and epochs affects performance, and you witnessed overfitting in real time.

**Part 2 — Bank Churn:** You compared a neural network against all four classical classifiers on the same dataset. The neural network won — but not by much. The real lesson was the SVM collapse and the accuracy vs. explainability tradeoff.

**Up Next — Group Lab:** You'll explore TensorFlow Playground, where you can *watch* a neural network learn in real time — adjust layers, neurons, and learning rate, and see the decision boundary form before your eyes.

**Next week (Chapter 10):** We teach machines to see — computer vision, powered by the same neural networks you just built.
